# Agentic workflow
- 2 Agents
    - Writer agent
        - tools
            - search tool
            - web scrape tool
            - @tool
    - Evaluator agent

## Loading the api key

In [ ]:
# loading api key
import dotenv
dotenv.load_dotenv(dotenv.find_dotenv())

## Model

In [1]:
# getting the model
import os
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model="deepseek-flash",
    temperature=0,
    max_tokens=None,
    max_retries=2,
    api_key=os.environ.get("DEEPSEEK"),
    extra_body={"thinking": {"type": "disabled"}},
)


## Tools
- search tool
- web scrape tool

In [2]:
import requests
from bs4 import BeautifulSoup
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults
import json


@tool
def search_web(query: str) -> str:
    """Search DuckDuckGo for web links and brief descriptions."""
    search_tool = DuckDuckGoSearchResults()
    try:
        return search_tool.run(query)
    except Exception as e:
        return f"Error occurred while searching: {str(e)}"


@tool
def scrape_web(url: str, start: int = 0, end: int = 3000) -> str:
    """
    Scrape and return plain text from a web page URL along with total character length.
    Use this when search snippets lack detailed information.

    Args:
        url: URL of the webpage to scrape.
        start: Starting character index (must be >= 0). Default is 0.
        end: Ending character index (must be >= start). Default is 3000.
    """
    try:
        if "bing.com/aclick" in url or "ad.doubleclick" in url:
            return "Error: Cannot scrape tracking/ad redirect URLs. Use a direct source URL."

        if start < 0 or end < start:
            return "Error: 'start' must be >= 0 and 'end' must be >= 'start'."

        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            )
        }

        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "noscript", "svg"]):
            tag.decompose()

        text = " ".join(soup.get_text(separator=" ", strip=True).split())
        total_length = len(text)

        return json.dumps({
            "text": text[start:end],
            "start": start,
            "end": min(end, total_length),
            "total_length": total_length,
        })
    except Exception as e:
        return f"Failed to scrape webpage: {e}"


/tmp/ipykernel_3428139/2028442273.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


## Agentic workflow

In [7]:
from langchain_core.messages import (
    SystemMessage, 
    HumanMessage, 
    AIMessage, 
    ToolMessage
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from typing import TypedDict, Annotated, Any
from operator import add
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
import logging 

logger = logging.getLogger(__name__)
if not logger.handlers:
    logger.setLevel(logging.DEBUG)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    ch = logging.StreamHandler()
    ch.setFormatter(formatter)
    logger.addHandler(ch)

MAX_TOOL_CALLS = 5
MAX_EVAL_CALLS = 3

tools = [search_web, scrape_web]
tool_mappings = {tool.name: tool for tool in tools}

# State defenition for the agentic workflow
class State(TypedDict):
    user_input: str
    history: Annotated[list, add]
    tool_call_iteration: int
    eval_iteration: int
    eval_passed: bool

state = State()

## Research Agent
## to answer questions based on the information available and tools provided.
research_system_message = f"""
You are a helpful assistant. Your duty is to answer user question based on the information you have.
If you don't know the answer, you should say "I don't know". Don't make up answers.
You are allowed to use tools, the following tools are available to you:
{'\n -'.join([f'{tool.name}: {tool.description}' for tool in tools])}
"""

research_prompt = ChatPromptTemplate.from_messages([
    ("system", research_system_message),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{user_input}")
])

def research_agent(state: State) -> AIMessage:
    tool_call_iteration = state.get("tool_call_iteration")
    logger.debug(f"tool call iteration: {tool_call_iteration}, evaluation iteration: {state.get('eval_iteration')}, evaluation passed: {state.get('eval_passed')}")
    if tool_call_iteration < MAX_TOOL_CALLS:
        llm_with_tools = llm.bind_tools(tools)
    else:
        llm_with_tools = llm
    chain = research_prompt | llm_with_tools
    response = chain.invoke(state)
    return {"history": [response]}

# tool call node
def tool_call_node(state: State) -> ToolMessage:
    last_response = state.get("history")[-1]
    iterations = state.get("tool_call_iteration", 0)

    iterations += 1
    tool_messages = []
    for tool_call in last_response.tool_calls:
        name = tool_call.get("name")
        args = tool_call.get("args")
        id = tool_call.get("id")
        logger.debug(f"Tool call: {name}, args: {args}, id: {id}")

        if name in tool_mappings:
            tool = tool_mappings[name]
            tool_response = tool.invoke(args)
            logger.debug(f"Tool response: {tool_response}")

            tool_messages.append(
                ToolMessage(content=tool_response, tool_call_id=id)
            )

    return {
        "history": tool_messages,
        "tool_call_iteration": iterations
    }

# Evaluation node
class EvaluationResult(BaseModel):
    passed: bool = Field(description="True if the answer is accurate, and fully answered the question")
    feedback: str = Field(description = "Feedback given by the user indicateing the shortcomings of the answer provided")

def evaluator(state: State) -> dict[bool, Any]:
    last_response = state.get("history")[-1]

    evaluation_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that evaluates the quality of the answer provided by the agent."),
        ("user", f"""
        Please evaluate the answer provided by the agent. 
        user input: {state.get("user_input")}
        agent answer: {last_response.content}
        Is it complete and accurate? If not, please provide feedback.
        """)
    ])
    llm_with_structured_output = llm.with_structured_output(EvaluationResult)
    chain = evaluation_prompt | llm_with_structured_output
    response = chain.invoke(state)
    result = response.model_dump()
    logger.debug(f"Evaluation result: {result}")
    return {
        "history": [AIMessage(content=f"Evaluation Result: Passed: {result.get('passed')}, Feedback: {result.get('feedback')}")],
        "tool_call_iteration": 0, # rest the tool call counter
        "eval_iteration": state.get("eval_iteration", 0) + 1, # count the evaluation
        "eval_passed": result.get("passed")
    }

def tool_cal_router(state: State) -> str:
    last_state = state.get("history")[-1]

    if last_state.tool_calls:
        return "tool_call_node"
    elif state.get("eval_iteration") < MAX_EVAL_CALLS and not state.get("eval_passed", False):
        return "evaluator"
    else:
        return END

graph = StateGraph(State)

graph.add_node("research_agent", research_agent)
graph.add_node("tool_call_node", tool_call_node)
graph.add_node("evaluator", evaluator)

graph.add_edge(START, "research_agent")
graph.add_edge("tool_call_node", "research_agent")
graph.add_edge("evaluator", "research_agent")

# conditional edge to tool_call and evaluator nodes
graph.add_conditional_edges(
    "research_agent", 
    tool_cal_router, 
    {
    "tool_call_node": "tool_call_node",
    "evaluator": "evaluator",
    END: END
    }
)  

app = graph.compile()

# user_input = "which is the best tourist place to visit in Frankfurt, Germany?"
user_input = "which is the most Beautiful place near frankfurt to have 2 days outing?"
response = app.invoke({
    "user_input": user_input,
    "history": [],
    "tool_call_iteration": 0,
    "eval_iteration": 0,
    "eval_passed": False
})
result = response.get("history")[-1]
print(result.content)


2026-11-20 16:52:15,912 - __main__ - DEBUG - tool call iteration: 0, evaluation iteration: 0, evaluation passed: False
2026-11-20 16:52:17,071 - __main__ - DEBUG - Tool call: search_web, args: {'query': 'most beautiful places near Frankfurt for a 2 day trip'}, id: call_00_e2Oqx3NXpL5Vg0jsAkK62104
2026-11-20 16:52:18,955 - __main__ - DEBUG - Tool response: snippet: Jul 17, 2026 · Are you looking for places to visit near Frankfurt? We have collected the best places near Frankfurt that you cannot miss!, title: Top 20 Unmissable Places to Visit Near Frankfurt - Top Travel ..., link: https://www.toptravelsights.com/places-near-frankfurt/, snippet: Jun 3, 2026 · Discover the best day trips from Frankfurt by train or car. Includes travel times for Heidelberg, the Rhine Valley, Eltz Castle, and local tips for cheap tickets., title: 15 Best Day Trips and Travel Tips from Frankfurt (2026), link: https://germanywander.com/day-trips-from-frankfurt, snippet: Apr 28, 2025 · Whether you’re craving me

Based on my research, the **most beautiful place near Frankfurt for a 2-day outing is the Romantic Rhine Valley (Rhine Gorge)** — specifically the stretch between **Rüdesheim and Koblenz**.

## Why it's the top choice
- It's a **UNESCO World Heritage** stretch of the Rhine, widely described as **one of Europe's most picturesque scenic getaways**.
- You get terraced vineyards, hilltop castles, and charming wine villages all in one compact area.
- It's easy to reach — about **1 hour from Frankfurt by train**.

## A simple 2-day plan
- **Day 1:** Train to Rüdesheim, take the cable car over the vineyards, do some wine tasting, then cruise or ride along the river toward Bacharach/St. Goar with castle stops.
- **Day 2:** Continue to Koblenz (where the Rhine meets the Moselle) or explore more riverside villages before heading back.

## Strong runners-up
| Place | Highlight | Travel time |
|-------|-----------|-------------|
| **Heidelberg** | Romantic castle ruins + old town | ~1 hr |
| **Rot